# **Bias-Variance Tradeoff**

Prerequisites: Probability/Statistics primer. Next: Regularization,
Cross-Validation.

## 1. Intuition

A model that's too simple (underfits) is consistently wrong in the same way
- **high bias**. A model that's too complex (overfits) is wildly sensitive
to which exact training sample it saw - **high variance**. Total error
decomposes cleanly into these two pieces plus noise you can never remove.

![bias-variance tradeoff](../_assets/bias_variance_tradeoff.png)

*(Generated by `../_scripts/plot_bias_variance_curve.py`.)*

## 2. Formal Setup

Assume true relationship $y = f(x) + \epsilon$, with $E[\epsilon]=0$,
$\text{Var}(\epsilon)=\sigma^2$. We fit $\hat f(x)$ on a training set $D$
(random). We want the **expected** test error at a fixed point $x_0$,
where the expectation is over the randomness of $D$ (which training set we
happened to draw) and $\epsilon$.

## 3. Full Derivation of the Decomposition

$$\text{Err}(x_0) = E_D\big[(y_0 - \hat f_D(x_0))^2\big]$$

Substitute $y_0 = f(x_0)+\epsilon_0$ and add/subtract $E_D[\hat f_D(x_0)]$:

$$y_0-\hat f_D(x_0) = \big(f(x_0)-E_D[\hat f_D(x_0)]\big) + \big(E_D[\hat f_D(x_0)]-\hat f_D(x_0)\big) + \epsilon_0$$

Call these three terms $A$ (bias term, constant w.r.t. $D$), $B$ (deviation
of a specific fit from the average fit), $\epsilon_0$ (noise, independent of
$D$). Squaring and taking $E_D[\cdot]$, all cross terms vanish because:
- $E_D[B] = 0$ by definition of $B$ (it's a deviation from its own mean)
- $\epsilon_0$ is independent of $D$ and has mean 0

$$\text{Err}(x_0) = \underbrace{\big(f(x_0)-E_D[\hat f_D(x_0)]\big)^2}_{\text{Bias}^2}
+ \underbrace{E_D\big[(\hat f_D(x_0)-E_D[\hat f_D(x_0)])^2\big]}_{\text{Variance}}
+ \underbrace{\sigma^2}_{\text{Irreducible noise}}$$

**Bias** = how far off the *average* prediction (over many resampled
training sets) is from the truth. **Variance** = how much the prediction
*swings* depending on which training set was drawn.

## 4. Worked Numerical Example

True function: $f(x) = 3x$ (no noise for simplicity, $\sigma^2=0$ so we
can isolate bias and variance exactly). Suppose we repeatedly draw tiny
2-point training sets and fit two candidate models: (a) a **constant**
predictor $\hat f(x) = \bar y$ (high bias, low variance) and (b) a
**degree-1 through the origin** predictor forced to fit exactly through
one point (high variance, low bias - extreme small-sample overfit).

Fix evaluation point $x_0=2$, true value $f(2)=6$. Draw three different
2-point training sets and look at the constant-predictor's prediction at
$x_0$:
- Set 1: points $(1,3),(3,9)$ → mean $\bar y = 6$ → predicts 6
- Set 2: points $(1,2.9),(3,9.2)$ → mean $\bar y = 6.05$ → predicts 6.05
- Set 3: points $(1,3.1),(3,8.8)$ → mean $\bar y = 5.95$ → predicts 5.95

$E_D[\hat f_D(2)] = (6+6.05+5.95)/3 = 6.0$.
$\text{Bias}^2 = (6 - 6.0)^2 = 0$ - this simple model is *unbiased* here
because $f(x)=3x$ happens to be symmetric around the sample mean at this
point, but its variance is:
$$\text{Var} = \tfrac13[(6-6)^2+(6.05-6)^2+(5.95-6)^2] = \tfrac13[0+0.0025+0.0025]=0.00167$$

Now compare to a high-degree polynomial forced through both points exactly
each time: predictions at $x_0=2$ would swing much more between resamples
(e.g., 5.7, 6.4, 5.5 depending on noise) - same exercise gives a much
larger Variance term, illustrating overfitting numerically rather than just
by definition.

## 5. Connection to Model Complexity (why the curve in §1 looks that way)

- Increasing model complexity (polynomial degree, tree depth, smaller $k$
  in KNN, smaller regularization strength) → **bias decreases**
  (model can represent the true function better) but **variance
  increases** (more parameters chase noise in the specific sample).
- The **optimal** complexity minimizes Bias² + Variance, not either alone.
  This is the single sentence that justifies: regularization (constrains
  variance), ensembling/bagging (reduces variance by averaging), boosting
  (reduces bias by sequentially correcting errors), and k in cross-validation.

## 6. Python - empirically demonstrating the decomposition

In [ ]:
import numpy as np

def true_fn(x): return np.sin(1.5 * x)

def experiment(degree, n_train=15, n_trials=200, noise_std=0.3, x0=1.0, seed=0):
    rng = np.random.default_rng(seed)
    preds = []
    for _ in range(n_trials):
        X = rng.uniform(-3, 3, n_train)
        y = true_fn(X) + rng.normal(0, noise_std, n_train)
        coeffs = np.polyfit(X, y, degree)
        preds.append(np.polyval(coeffs, x0))
    preds = np.array(preds)
    bias2 = (true_fn(x0) - preds.mean())**2
    variance = preds.var()
    return bias2, variance

for deg in [1, 3, 6, 9]:
    b2, v = experiment(deg)
    print(f"degree={deg:2d}  bias^2={b2:.4f}  variance={v:.4f}  total={b2+v:.4f}")

degree= 1  bias^2=0.9756  variance=0.0576  total=1.0333
degree= 3  bias^2=0.0755  variance=0.0366  total=0.1121
degree= 6  bias^2=0.0007  variance=0.2295  total=0.2302
degree= 9  bias^2=0.3088  variance=116.4546  total=116.7634


Bias² falls sharply from degree 1→3 (the linear model badly underfits a
sine curve), stays low through moderate complexity, then variance
explodes at degree 9 as the polynomial starts fitting individual noisy
points rather than the underlying curve - the exact pattern in the
generated figure above. (Very high degrees, e.g. 15, become numerically
unstable - `np.polyfit` itself warns about ill-conditioning - which is a
realistic illustration of *why* extremely flexible unregularized models
are avoided in practice, not just a variance-in-theory statement.)